# Surface Example — GlyphViz DEM Generator (Cowles Mountain)

This notebook generates the node + tag CSVs for a real digital elevation
model (DEM) rendered with GlyphViz's `TOPO_SURFACE` (`glyphviz_core/topology.py`):
**Cowles Mountain** (San Diego, CA), built from two co-registered,
identically-sized JPGs in this folder:

- `cowles_dem_lowres.jpg` — a grayscale heightmap (brighter = higher elevation)
- `cowles_aerial_lowres.jpg` — the matching aerial photo, used only for per-point color

No new dependency is needed for JPEG decoding — `PySide6.QtGui.QImage` (an
existing GlyphViz dependency, the same class `glyphviz_gl/texture_manager.py`
already uses for textures) reads both JPGs directly, without a `QApplication`
instance.

## How TOPO_SURFACE works

Children of a Surface-topology parent are placed at their literal Cartesian
`(translate_x, translate_y)` — the grid position — with `translate_z` as the
height at that grid point (see `_surface_offset` in `glyphviz_core/topology.py`).
`glyphviz_gl/viewport.py`'s `_draw_topology_overlays` groups a Surface
parent's children back into a grid purely by their distinct
`(translate_x, translate_y)` values (no explicit row/col column needed) and
draws a lit, per-vertex-colored `GL_QUADS` mesh between them — so each
child's own `color_r/g/b/a` becomes that grid point's mesh color for free.
Each child also gets its own small `GEO_POINT` glyph (to keep the point
count cheap to render), which picks up the same color.

## Output files

- `Surface_Example_gv_node.csv` — the Surface root + one child per sampled grid point
- `Surface_Example_gv_tag.csv` — one label on the root node

## Loading in GlyphViz

**File > Open Node CSV** → `Surface_Example_gv_node.csv`.

In [ ]:
import csv
from pathlib import Path

from PySide6.QtGui import QImage

## Parameters

Edit the values in this cell to customise the example — most importantly
**`STRIDE`**, which controls resolution: it samples every `STRIDE`'th pixel
in both images. The default (10) is deliberately crude, per the original
ask — a manageable ~1,400-point grid with good FPS, where the parent scales
correctly and individual child points are still easy to select/edit
in the GlyphViz GUI. Lower it (e.g. 5 or 2) once this crude pass looks
right and you want more detail; the source images are 390x369, so `STRIDE=1`
would sample every pixel (~144,000 points — likely too many for smooth
interaction, but nothing stops you from trying).

In [ ]:
OUTPUT_DIR = Path('.').resolve()
PREFIX     = 'Surface_Example'

DEM_JPG    = OUTPUT_DIR / 'cowles_dem_lowres.jpg'
AERIAL_JPG = OUTPUT_DIR / 'cowles_aerial_lowres.jpg'

TOPO_SURFACE = 17   # glyphviz_core/topology.py
GEO_POINT    = 22   # glyphviz_core/geometry_data.py

STRIDE       = 10    # <-- sample every STRIDE'th pixel in both images (resolution knob)
EXAGGERATION = 12.0  # world-unit height at full-white (255) grayscale
CHILD_SCALE  = 0.15  # small GEO_POINT glyph size (world units) per grid point

## Sampling the images

`sample_height()` maps a grayscale value (0-255) to a world-unit height.
**Elevation convention assumed: brighter pixel = higher elevation** —
confirmed against the source image (the single brightest region is an
isolated peak-like blob, consistent with a real DEM render rather than
noise). If the rendered terrain looks inverted once viewed in the app,
negate the fraction in `sample_height()` below.

`build_grid()` walks both images at `STRIDE` spacing and returns, for every
sampled grid point, its column/row index, exaggerated height, and the
aerial photo's RGB at that same pixel — asserting the two images are the
same pixel size first, since they must be co-registered for this to make
sense.

In [ ]:
def sample_height(gray_0_255: float, exaggeration: float) -> float:
    """Brighter = higher. See the Markdown cell above if this needs inverting."""
    return (gray_0_255 / 255.0) * exaggeration


def build_grid(dem: QImage, aerial: QImage, stride: int, exaggeration: float):
    """Returns list of (col_idx, row_idx, translate_z, color_r, color_g, color_b)
    for every sampled grid point, in row-major order, row 0 = north (top of
    the source images) so that translate_y increases northward once flipped
    by the caller."""
    w, h = dem.width(), dem.height()
    assert (w, h) == (aerial.width(), aerial.height()), \
        'DEM and aerial images must be the same pixel size (co-registered)'

    rows = list(range(0, h, stride))
    cols = list(range(0, w, stride))
    points = []
    for ri, y in enumerate(rows):
        for ci, x in enumerate(cols):
            dem_c = dem.pixelColor(x, y)
            gray = (dem_c.red() + dem_c.green() + dem_c.blue()) / 3.0
            tz = sample_height(gray, exaggeration)

            aerial_c = aerial.pixelColor(x, y)
            points.append((ci, ri, tz, aerial_c.red(), aerial_c.green(), aerial_c.blue()))
    return points, len(cols), len(rows)

## 1 · Node file (`gv_node.csv`)

Key fields:
- `topo` — `TOPO_SURFACE` on the root only; children leave it at 0 (they have no children of their own).
- `translate_x/y` — grid position (grid-index units); `translate_z` — exaggerated DEM height.
- `geometry` — `GEO_POINT` for every node (root and children), per the original ask to keep this cheap to render.
- `color_r/g/b` — the aerial photo's color at that grid point (root just gets a neutral gray).

**Gotcha hit while building this**: `glyphviz_gl/viewport.py`'s quad-mesh
renderer gates the entire mesh draw on the *Surface parent's own* `hide`
flag, not just its own glyph. An initial attempt to hide the structural
root node (wanting it invisible) silently skipped the whole mesh, not just
the root's point. Keep `hide=0` on the root.

Similarly, the root's `scale_x/y/z` must stay `1.0` — `_surface_offset`
multiplies every child's `translate_x/y/z` by the *parent's* world scale,
so that's what makes 1 grid-index unit = 1 world unit. This does mean the
root's own `GEO_POINT` glyph renders a little larger than the children's
(whose `scale_x/y/z` is the separate, smaller `CHILD_SCALE`) — but it's one
dot among ~1,400, sitting near the grid's centroid, and isn't visually
intrusive in practice (confirmed via headless render).

In [ ]:
def write_nodes(stride: int, exaggeration: float, child_scale: float):
    dem = QImage(str(DEM_JPG))
    aerial = QImage(str(AERIAL_JPG))
    if dem.isNull() or aerial.isNull():
        raise SystemExit(f'Could not load {DEM_JPG.name} / {AERIAL_JPG.name}')

    points, n_cols, n_rows = build_grid(dem, aerial, stride, exaggeration)
    print(f'Sampled {n_cols}x{n_rows} = {len(points)} grid points '
          f'(stride={stride}) from {dem.width()}x{dem.height()} source images')

    # Center the grid horizontally on the root (translate_x/y in grid-index
    # units, root scale_x/y = 1.0 -> 1 world unit per grid cell); flip row
    # order so translate_y increases northward (row 0 in the image = north).
    x_off = -(n_cols - 1) / 2.0
    y_off = -(n_rows - 1) / 2.0

    fieldnames = [
        'id', 'type', 'parent_id', 'branch_level',
        'translate_x', 'translate_y', 'translate_z',
        'rotate_x', 'rotate_y', 'rotate_z',
        'scale_x', 'scale_y', 'scale_z',
        'color_r', 'color_g', 'color_b', 'color_a',
        'geometry', 'hide', 'topo', 'ratio',
    ]
    rows = [{
        'id': 1, 'type': 5, 'parent_id': 0, 'branch_level': 0,
        'translate_x': 0.0, 'translate_y': 0.0, 'translate_z': 0.0,
        'rotate_x': 0.0, 'rotate_y': 0.0, 'rotate_z': 0.0,
        'scale_x': 1.0, 'scale_y': 1.0, 'scale_z': 1.0,
        'color_r': 128, 'color_g': 128, 'color_b': 128, 'color_a': 255,
        'geometry': GEO_POINT, 'hide': 0, 'topo': TOPO_SURFACE, 'ratio': 0.1,
    }]
    next_id = 2
    for ci, ri, tz, r, g, b in points:
        rows.append({
            'id': next_id, 'type': 5, 'parent_id': 1, 'branch_level': 1,
            'translate_x': round(ci + x_off, 4),
            'translate_y': round((n_rows - 1 - ri) + y_off, 4),
            'translate_z': round(tz, 4),
            'rotate_x': 0.0, 'rotate_y': 0.0, 'rotate_z': 0.0,
            'scale_x': child_scale, 'scale_y': child_scale, 'scale_z': child_scale,
            'color_r': r, 'color_g': g, 'color_b': b, 'color_a': 255,
            'geometry': GEO_POINT, 'hide': 0, 'topo': 0, 'ratio': 0.1,
        })
        next_id += 1

    path = OUTPUT_DIR / f'{PREFIX}_gv_node.csv'
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f'Written: {path.name}  ({len(rows)} nodes: 1 root + {len(points)} DEM points)')
    return points, n_cols, n_rows


points, n_cols, n_rows = write_nodes(STRIDE, EXAGGERATION, CHILD_SCALE)

## 2 · Tag file (`gv_tag.csv`)

One label on the root node, noting the stride and exaggeration used —
useful once you're comparing multiple regenerated versions side by side.

In [ ]:
def write_tags(stride: int, exaggeration: float):
    path = OUTPUT_DIR / f'{PREFIX}_gv_tag.csv'
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['id', 'table_id', 'record_id', 'title'])
        w.writeheader()
        w.writerow({
            'id': 0, 'table_id': 0, 'record_id': 1,
            'title': f'Cowles Mountain DEM (stride={stride}, {exaggeration:.0f}x height)',
        })
    print(f'Written: {path.name}  (1 label on the root node)')


write_tags(STRIDE, EXAGGERATION)

## Preview — sanity-check against the real placement math

Re-derives a couple of sample points' world offsets directly via
`glyphviz_core.topology._surface_offset` (the same function GlyphViz uses
at render time) as a check that the CSV's numbers mean what this notebook
thinks they mean.

In [ ]:
import sys
sys.path.insert(0, str(Path('../..').resolve()))
from glyphviz_core import topology as T

root_world_scale = (1.0, 1.0, 1.0)   # matches scale_x/y/z written for the root above
for ci, ri, tz, r, g, b in points[:3] + points[-1:]:
    tx = ci - (n_cols - 1) / 2.0
    ty = (n_rows - 1 - ri) - (n_rows - 1) / 2.0
    world = T._surface_offset(tx, ty, tz, 0.0, 0.1, root_world_scale)
    print(f'grid ({ci:>3},{ri:>3})  height={tz:6.3f}  -> world {tuple(round(c, 3) for c in world)}')

print(f'\nheight range: {min(p[2] for p in points):.3f} .. {max(p[2] for p in points):.3f}')

## Adapting this example

| Goal | What to change |
|------|---------------|
| More/fewer points (resolution) | `STRIDE` — lower = more detail, more points |
| Taller/flatter relief | `EXAGGERATION` |
| Bigger/smaller point glyphs | `CHILD_SCALE` |
| A different mountain/terrain | Swap in a different co-registered DEM/aerial JPG pair (must be the same pixel size as each other) and update `DEM_JPG`/`AERIAL_JPG` |
| Inverted-looking terrain | Negate the fraction in `sample_height()` |
| Command-line use instead of a notebook | `generate_surface_example.py` exposes `STRIDE`/`EXAGGERATION`/`CHILD_SCALE` as `--stride`/`--exaggeration`/`--child-scale` CLI flags |